# Ch5. The Forecaster's Toolbox
**Forecasting: Principles & Practice (Python Edition)**
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast

## [Slide 3] 5.1 Toolbox Imports

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import (
    HistoricAverage, Naive,
    RandomWalkWithDrift, SeasonalNaive, SklearnModel,
)
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mape, mase

## [Slide 5] 5.1 The Six-Step Workflow

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
gdp_df = (
    pd.read_csv(BASE + "global_economy.csv",
                parse_dates=["ds"])
    [["unique_id","ds","GDP","Population"]]
    .assign(
        GDP=lambda x: x["GDP"].interpolate(),
        Population=lambda x: x["Population"].interpolate(),
        y=lambda x: x["GDP"] / x["Population"],
    )
)

## [Slide 8] 5.1 Specify and Estimate

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import SklearnModel
from utilsforecast.feature_engineering import pipeline, trend
from fpppy.models import LinearRegression

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
gdp_df = (
    pd.read_csv(BASE + "global_economy.csv", parse_dates=["ds"])
    [["unique_id","ds","GDP","Population"]]
    .assign(
        GDP=lambda x: x["GDP"].interpolate(),
        Population=lambda x: x["Population"].interpolate(),
        y=lambda x: x["GDP"] / x["Population"],
    )
)
sweden_df = (
    gdp_df
    .loc[lambda x: x["unique_id"] == "Sweden"]
    [["unique_id","ds","y"]]
)
# Create trend feature
train_features, valid_features = pipeline(
    sweden_df, features=[trend], freq="Y", h=3
)
# Fit linear trend model
trend_model = SklearnModel(LinearRegression())
sf = StatsForecast(models=[trend_model], freq="Y")
sf.fit(df=train_features)

In [ ]:
fcasts = sf.predict(h=3, X_df=valid_features)
model = sf.fitted_[0][0].model_["model"]
fcasts = model.add_prediction_intervals(
    fcasts, valid_features.rename(
        columns={"trend": "x1"}))
print(fcasts.round(1))

## [Slide 16] 5.2 Benchmark Methods in Python

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import (
    HistoricAverage, Naive, SeasonalNaive, RandomWalkWithDrift,
)

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv",
    parse_dates=["ds"]
)
train = production_df.loc[
    lambda x: (x["unique_id"]=="Beer") &
              (x["ds"] < "2007")
]
test = production_df.loc[
    lambda x: (x["unique_id"]=="Beer") &
              (x["ds"] >= "2007")
]

sf = StatsForecast(
    models=[
        HistoricAverage(),    # Mean
        Naive(),              # Naive
        SeasonalNaive(4),     # Seasonal Naive (q=4)
        RandomWalkWithDrift() # Drift
    ],
    freq=pd.offsets.QuarterBegin(1)
)
sf.fit(train)
fcasts = sf.predict(h=14).assign(y=test["y"].to_numpy())
print(fcasts.round(1))

## [Slide 20] 5.2 Benchmark Forecasts — Chart

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import (
    HistoricAverage, Naive, SeasonalNaive, RandomWalkWithDrift,
)

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv", parse_dates=["ds"]
)
train = production_df.loc[
    lambda x: (x["unique_id"]=="Beer") & (x["ds"] < "2007")
]
test = production_df.loc[
    lambda x: (x["unique_id"]=="Beer") & (x["ds"] >= "2007")
]
sf = StatsForecast(
    models=[HistoricAverage(), Naive(), SeasonalNaive(4), RandomWalkWithDrift()],
    freq=pd.offsets.QuarterBegin(1),
)
sf.fit(train)
fcasts = sf.predict(h=14).assign(y=test["y"].to_numpy())

fig, ax = plt.subplots(figsize=(8, 3.4))
last16 = train.iloc[-16:]
ax.plot(last16["ds"], last16["y"], 'k-', lw=1.5, label='Training data')
ax.plot(fcasts["ds"], fcasts["y"], 'k--', lw=1, alpha=0.4, label='Actual (test)')
ax.plot(fcasts["ds"], fcasts["HistoricAverage"], 's-', ms=4, label='Mean')
ax.plot(fcasts["ds"], fcasts["Naive"], 'o-', ms=4, label='Naïve')
ax.plot(fcasts["ds"], fcasts["SeasonalNaive"], '^-', ms=4, label='Seasonal Naïve')
ax.plot(fcasts["ds"], fcasts["RWD"], 'v-', ms=4, label='Drift')
ax.axvline(train["ds"].iloc[-1], color='gray', lw=0.8, ls=':')
ax.legend(fontsize=7, ncol=3)
ax.set(title='Four benchmark forecasts on Beer production (real data)',
       xlabel='Quarter', ylabel='Megalitres')
plt.tight_layout(); plt.show()

## [Slide 24] 5.3 Fitted Values and Residuals

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import HistoricAverage

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv", parse_dates=["ds"]
)
train = production_df.loc[
    lambda x: (x["unique_id"]=="Beer") & (x["ds"] < "2007")
]

sf = StatsForecast(
    models=[HistoricAverage()], freq="Q"
)
sf.forecast(h=14, df=train, fitted=True)
fitted_values = sf.forecast_fitted_values()

df = train.assign(
    fitted=fitted_values["HistoricAverage"].to_numpy(),
    resid=lambda x: x["y"] - x["fitted"],
)
print(df.tail())

## [Slide 29] 5.4 Diagnostic Plots in Python

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import Naive
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
goog_df = pd.read_csv(
    BASE + "gafa_stock.csv", parse_dates=["ds"]
).loc[lambda x: x["unique_id"] == "GOOG_Close"]

sf = StatsForecast(models=[Naive()], freq="B")
sf.forecast(h=1, df=goog_df, fitted=True)
fitted_values = sf.forecast_fitted_values()
resid = goog_df.reset_index(drop=True).assign(
    resid=lambda x: x["y"] - fitted_values["Naive"].to_numpy()
)
# subset=["resid"]: 이 컬럼의 NaN만 검사해 행 제거
resid = resid.dropna(subset=["resid"])

# Ljung-Box test (lag 10, non-seasonal daily data)
lb_test = acorr_ljungbox(
    resid["resid"], boxpierce=False, lags=[10]
)
print(lb_test.round(4))

fig, axes = plt.subplot_mosaic(
    [["resid","resid"],["acf","hist"]], figsize=(9, 4.2)
)
axes["resid"].plot(resid["ds"], resid["resid"], lw=1)
axes["resid"].set(title="Innovation Residuals (Naïve, GOOG closing price)")
plot_acf(resid["resid"], zero=False,
         bartlett_confint=False, ax=axes["acf"], marker="None")
axes["acf"].set_title("ACF of Residuals")
axes["hist"].hist(resid["resid"], bins=20,
                  color="steelblue", edgecolor="white")
axes["hist"].set_title("Histogram of Residuals")
plt.tight_layout(); plt.show()

## [Slide 36] 5.5 Seasonal Naïve Prediction Intervals — Python

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv", parse_dates=["ds"]
)
train = production_df.loc[
    lambda x: (x["unique_id"]=="Beer") & (x["ds"] < "2007")
]
sf = StatsForecast(
    models=[SeasonalNaive(4)], freq=pd.offsets.QuarterBegin(1)
)
fcasts = sf.forecast(df=train, h=8, level=[80, 95])

summary = pd.DataFrame({
    "h": np.arange(1, 9),
    "k": np.arange(0, 8) // 4,
    "forecast": fcasts["SeasonalNaive"].round(1).to_numpy(),
    "width80": (fcasts["SeasonalNaive-hi-80"] - fcasts["SeasonalNaive-lo-80"]).round(1).to_numpy(),
    "width95": (fcasts["SeasonalNaive-hi-95"] - fcasts["SeasonalNaive-lo-95"]).round(1).to_numpy(),
})
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.4))
last16 = train.iloc[-16:]
ax.plot(last16["ds"], last16["y"], "k-", label="Training data")
ax.plot(fcasts["ds"], fcasts["SeasonalNaive"], "o-",
        color="darkorange", label="Seasonal Naïve forecast")
ax.fill_between(fcasts["ds"], fcasts["SeasonalNaive-lo-95"],
                 fcasts["SeasonalNaive-hi-95"], color="darkorange",
                 alpha=0.2, label="95% PI")
ax.fill_between(fcasts["ds"], fcasts["SeasonalNaive-lo-80"],
                 fcasts["SeasonalNaive-hi-80"], color="darkorange",
                 alpha=0.35, label="80% PI")
ax.axvline(train["ds"].iloc[-1], color="gray", lw=0.8, ls=":")
ax.legend(fontsize=7, ncol=2)
ax.set(title="Seasonal Naïve forecast with prediction intervals (Beer, h=8)")
plt.tight_layout(); plt.show()

## [Slide 40] 5.5 Bootstrap Prediction Intervals

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import Naive

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv", parse_dates=["ds"]
)
train = production_df.loc[
    lambda x: (x["unique_id"]=="Beer") & (x["ds"] < "2007")
]

sf = StatsForecast(models=[Naive()], freq=pd.offsets.QuarterBegin(1))
sf.forecast(h=30, df=train, fitted=True)
fitted = sf.forecast_fitted_values()
errors = (fitted["y"] - fitted["Naive"]).iloc[1:].to_numpy()
errors -= errors.mean()  # Centre residuals
last_obs = train["y"].iloc[-1]

# def name(params): reusable function; return sends the value back
def generate_sim_df(num_sims, h):
    steps = np.random.choice(errors, size=(num_sims, h))
    sims = last_obs + steps.cumsum(axis=1)
    return pd.DataFrame(
        sims.T,
        columns=[f"sim_{i}" for i in range(num_sims)]
    )

# Bootstrap quantiles from 5000 simulations
h = 30
raw_df = generate_sim_df(num_sims=5000, h=h)
lo95 = raw_df.quantile(0.025, axis="columns")
hi95 = raw_df.quantile(0.975, axis="columns")
lo80 = raw_df.quantile(0.10, axis="columns")
hi80 = raw_df.quantile(0.90, axis="columns")
future = pd.date_range(train["ds"].iloc[-1], periods=h + 1,
                        freq=pd.offsets.QuarterBegin(1))[1:]

fig, ax = plt.subplots(figsize=(7, 3.2))
last16 = train.iloc[-16:]
ax.plot(last16["ds"], last16["y"], "k-", label="Training data")
ax.plot(future, raw_df.median(axis="columns"), "o-",
        color="darkorange", label="Median simulated path")
ax.fill_between(future, lo95, hi95, color="darkorange", alpha=0.2, label="95% PI")
ax.fill_between(future, lo80, hi80, color="darkorange", alpha=0.35, label="80% PI")
ax.legend(fontsize=7, ncol=2)
ax.set(title="Bootstrap prediction intervals (Naïve, Beer production, 5000 sims)")
plt.tight_layout(); plt.show()

## [Slide 47] 5.6 Variance Stabilization — Chart

In [ ]:
import numpy as np, matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n = 120
t = np.arange(n)
y_het = (10 * np.exp(0.03*t)
         * (1 + 0.25*np.sin(2*np.pi*t/12))
         + rng.normal(0, 0.5, n) * np.exp(0.015*t))
y_het = np.clip(y_het, 1e-3, None)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].plot(t, y_het, lw=1, color="steelblue")
axes[0].set(title="Original series\n(heteroscedastic)", xlabel="t")

axes[1].plot(t, np.sqrt(y_het), lw=1, color="darkorange")
axes[1].set(title="Square-root transform", xlabel="t")

axes[2].plot(t, np.log(y_het), lw=1, color="green")
axes[2].set(title="Log transform\n(stabilized)", xlabel="t")

plt.suptitle("Variance Stabilization: Synthetic Exponential Growth Series",
             fontsize=9)
plt.tight_layout(); plt.show()

## [Slide 49] 5.6 Automated Lambda Selection

In [ ]:
import numpy as np
from scipy.stats import boxcox, boxcox_normmax

rng = np.random.default_rng(7)
n = 120
t = np.arange(n)
y_pos = (10 * np.exp(0.03*t)
         * (1 + 0.25*np.sin(2*np.pi*t/12))
         + 0.5 * rng.exponential(1, n))
y_pos = np.clip(y_pos, 0.01, None)

# Estimate optimal lambda
lambda_opt = boxcox_normmax(y_pos, method="mle")
print(f"Optimal lambda (MLE): {lambda_opt:.4f}")

# Transform with estimated lambda
y_transformed, lambda_fit = boxcox(y_pos)
print(f"Lambda used by boxcox(): {lambda_fit:.4f}")

# Manual Box-Cox for general forecasting pipeline
def boxcox_transform(y, lam):
    if abs(lam) < 1e-6:
        return np.log(y)
    return (np.sign(lam*y + 1) *
            np.abs(lam*y + 1)**(1/lam))

## [Slide 53] 5.6 Bias Adjustment for Back-Transformed Forecasts

In [ ]:
import numpy as np, pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import RandomWalkWithDrift

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv", parse_dates=["ds"]
)
# Forecast on the log scale (variance-stabilizing transform, lambda = 0)
train = (
    production_df
    .loc[lambda x: (x["unique_id"]=="Beer") & (x["ds"] < "2007")]
    .assign(y=lambda x: np.log(x["y"]))
)

sf = StatsForecast(models=[RandomWalkWithDrift()], freq=pd.offsets.QuarterBegin(1))
sf.forecast(h=8, df=train, fitted=True)
fitted = sf.forecast_fitted_values()
resid = (fitted["y"] - fitted["RWD"]).iloc[1:].to_numpy()
sigma_hat = resid.std(ddof=1)

T = len(train)
h = np.arange(1, 9)
sigma_h = sigma_hat * np.sqrt(h * (1 + h / (T - 1)))  # Drift PI formula, Sec. 5.5

fcasts_log = sf.forecast(df=train, h=8)

# Bias-adjusted back-transformation (lambda = 0, i.e. log)
fcasts = fcasts_log.assign(
    RWD_adjusted=np.exp(fcasts_log["RWD"]) *
                 (1 + sigma_h**2 / 2)
)
print(fcasts[["ds","RWD","RWD_adjusted"]].round(2).to_string(index=False))

## [Slide 55] 5.7 Decomposition-Based Forecasting

In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import STL

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
emp = pd.read_csv(BASE + "us_employment.csv", parse_dates=["ds"])
us_retail_employment_df = (
    emp.loc[lambda x: (x["unique_id"]=="Retail Trade") & (x["ds"] >= "1990")]
    [["unique_id","ds","y"]]
)

stl = STL(us_retail_employment_df["y"].to_numpy(),
           period=12, robust=True)
res = stl.fit()
train_df = us_retail_employment_df.assign(
    seasonal=res.seasonal,
    y_adjusted=lambda x: x["y"] - x["seasonal"],
)

## [Slide 58] 5.7 Decomposition Forecasting: Full Example

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive
from statsmodels.tsa.seasonal import STL

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
emp = pd.read_csv(BASE + "us_employment.csv", parse_dates=["ds"])
us_retail_employment_df = (
    emp.loc[lambda x: (x["unique_id"]=="Retail Trade") & (x["ds"] >= "1990")]
    [["unique_id","ds","y"]]
)
stl = STL(us_retail_employment_df["y"].to_numpy(), period=12, robust=True)
res = stl.fit()
train_df = us_retail_employment_df.assign(
    seasonal=res.seasonal,
    y_adjusted=lambda x: x["y"] - x["seasonal"],
)

# Forecast the seasonally adjusted component
sf_naive = StatsForecast(models=[Naive()], freq="M")
adj_fcasts = sf_naive.forecast(
    h=24, level=[80,95],
    df=train_df, target_col="y_adjusted"
)
# Forecast the seasonal component
sf_seasonal = StatsForecast(
    models=[SeasonalNaive(12, alias="Seasonal")], freq="M"
)
seas_fcasts = sf_seasonal.forecast(
    h=24, level=[80,95],
    df=train_df, target_col="seasonal"
)
# Combine
fcasts = adj_fcasts.merge(
    seas_fcasts, on=["unique_id","ds"]
).assign(
    combined=lambda x: x["Naive"] + x["Seasonal"]
)
print(fcasts[["ds","Naive","Seasonal","combined"]].head())

## [Slide 63] 5.8 Accuracy Evaluation in Python

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import (
    HistoricAverage, Naive, SeasonalNaive, RandomWalkWithDrift,
)
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mase
from functools import partial

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv", parse_dates=["ds"]
)
beers_df = production_df.loc[lambda x: x["unique_id"]=="Beer"]
train_df = beers_df.loc[lambda x: x["ds"].dt.year <= 2007]
test_df = beers_df.loc[lambda x: x["ds"].dt.year > 2007]

sf = StatsForecast(
    freq=pd.offsets.QuarterBegin(1),
    models=[HistoricAverage(), Naive(),
            SeasonalNaive(4), RandomWalkWithDrift()]
)
preds = sf.forecast(df=train_df, h=10).assign(y=test_df["y"].to_numpy())
evaluation = evaluate(
    preds, train_df=train_df,
    metrics=[rmse, mae, partial(mase, seasonality=4)]
)
print(evaluation.drop(columns="unique_id").round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(train_df["ds"].iloc[-16:], train_df["y"].iloc[-16:], "k-", label="Training")
axes[0].plot(test_df["ds"], test_df["y"], "k--", alpha=0.5, label="Actual (test)")
for m in ["HistoricAverage","Naive","SeasonalNaive","RWD"]:
    axes[0].plot(preds["ds"], preds[m], "o-", ms=3, label=m)
axes[0].legend(fontsize=6, ncol=2); axes[0].set_title("Forecasts vs. actual (Beer)")

metric_names = evaluation["metric"]
axes[1].bar(
    np.arange(len(metric_names)) - 0.2,
    evaluation.set_index("metric")["SeasonalNaive"], width=0.4, label="SeasonalNaive"
)
axes[1].bar(
    np.arange(len(metric_names)) + 0.2,
    evaluation.set_index("metric")["RWD"], width=0.4, label="RWD"
)
axes[1].set_xticks(np.arange(len(metric_names)), metric_names)
axes[1].legend(fontsize=7); axes[1].set_title("Accuracy by metric")
plt.tight_layout(); plt.show()

## [Slide 69] 5.9 CRPS in Python

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import HistoricAverage, Naive, RandomWalkWithDrift
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mqloss

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
production_df = pd.read_csv(
    BASE + "aus_production_formatted.csv", parse_dates=["ds"]
)
beers_df = production_df.loc[lambda x: x["unique_id"]=="Beer"]
train_df = beers_df.loc[lambda x: x["ds"].dt.year <= 2007]
test_df = beers_df.loc[lambda x: x["ds"].dt.year > 2007]

sf = StatsForecast(
    freq=pd.offsets.QuarterBegin(1),
    models=[HistoricAverage(), Naive(), RandomWalkWithDrift()]
)
levels = list(np.arange(0.1, 100, 1))  # exclusive of 0 and 100
preds = sf.forecast(df=train_df, h=len(test_df), level=levels).assign(
    y=test_df["y"].to_numpy()
)

models = ["RWD", "HistoricAverage", "Naive"]
raw_crps = evaluate(df=preds, models=models, metrics=[mqloss], level=levels)
# CRPS = 2 * mean quantile loss
crps_series = 2 * raw_crps[models].iloc[0]

# Skill score relative to Naive
naive_crps = crps_series.loc["Naive"]
skill = (naive_crps - crps_series) / naive_crps

print("CRPS:"); print(crps_series.round(3).to_string())
print("\nSkill vs. Naive:"); print(skill.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(crps_series.index, crps_series.to_numpy(), color="steelblue")
axes[0].set_title("CRPS by model (Beer)")
axes[1].bar(skill.index, skill.to_numpy(),
            color=["gray" if m=="Naive" else "darkorange" for m in skill.index])
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Skill score vs. Naive")
plt.tight_layout(); plt.show()

## [Slide 73] 5.10 Cross-Validation in Python

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import RandomWalkWithDrift
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
goog_df = pd.read_csv(
    BASE + "gafa_stock.csv", parse_dates=["ds"]
).loc[lambda x: x["unique_id"] == "GOOG_Close"]
goog_2015 = goog_df.loc[
    lambda x: x["ds"].dt.year == 2015
].copy()

drift_method = RandomWalkWithDrift()
sf = StatsForecast(models=[drift_method], freq="B")
cv_df = sf.cross_validation(
    h=1, df=goog_2015,
    step_size=1, test_size=249
)

# Evaluate across all validation windows (pool cutoffs first)
cv_eval = evaluate(cv_df.drop(columns="cutoff"),
                    metrics=[rmse], models=["RWD"])
print(cv_eval)

## [Slide 76] 5.10 Cross-Validation Across Forecast Horizons

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import RandomWalkWithDrift
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
goog_df = pd.read_csv(
    BASE + "gafa_stock.csv", parse_dates=["ds"]
).loc[lambda x: x["unique_id"] == "GOOG_Close"]
goog_2015 = goog_df.loc[lambda x: x["ds"].dt.year == 2015].copy()

rmses = []
horizons = np.arange(1, 9)
for h in horizons:
    sf = StatsForecast(
        models=[RandomWalkWithDrift()], freq="B"
    )
    cv_df = sf.cross_validation(
        h=h, df=goog_2015,
        step_size=1, test_size=249
    )
    cv_eval = evaluate(cv_df.drop(columns="cutoff"),
                        metrics=[rmse], models=["RWD"])
    rmses.append(cv_eval["RWD"].iloc[0])

print(pd.DataFrame({"h": horizons, "RMSE": np.round(rmses, 3)}).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.scatter(horizons, rmses, s=60, zorder=5)
ax.plot(horizons, rmses, lw=1, alpha=0.5)
ax.set(title="RMSE vs. Forecast Period (Google 2015, Drift method)",
       xlabel="Forecast period h", ylabel="Cross-validated RMSE")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## [Slide 78] 5.10 Cross-Validation — cv_df의 차원

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import RandomWalkWithDrift

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
goog_df = pd.read_csv(
    BASE + "gafa_stock.csv", parse_dates=["ds"]
).loc[lambda x: x["unique_id"] == "GOOG_Close"]
goog_2015 = goog_df.loc[lambda x: x["ds"].dt.year == 2015].copy()

for h_ in [1, 4, 8]:
    sf_ = StatsForecast(
        models=[RandomWalkWithDrift()], freq="B"
    )
    cv_df_ = sf_.cross_validation(
        h=h_, df=goog_2015, step_size=1, test_size=249
    )
    print(f"h={h_}: cv_df.shape = {cv_df_.shape}")